# Customer Churn Prediction

## Problem Statement

Customer churn refers to when a customer stops using a company's service.

In this project, we will build a machine learning model to predict whether a customer is likely to churn based on their usage and account information.

This is a **binary classification problem** where:

- 0 → Customer stays
- 1 → Customer churns

The goal is to help businesses identify customers who are at risk of leaving,
so they can take actions to retain them.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("./Data/Churn.csv")
df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())

In [ ]:
df['Churn'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
df['TotalCharges'].isnull().sum()
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [ ]:
df.isna().sum()
df.dtypes

## Encoding Categorical Features

Machine learning models cannot work with categorical (text) data.

Therefore, we convert categorical variables into numerical form.

Steps:

1. Convert binary categorical variables into 0 and 1
2. Apply encoding to multi-category features
3. Convert target variable (Churn) into numeric form

In [ ]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})

In [ ]:
binary_cols = [
    'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling'
]

for col in binary_cols:
    df[col] = df[col].map({'Yes':1, 'No':0})

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
df.sample()

### EDA

#### Checking the relation of churn column with others

## Churn Distribution

In [ ]:
sns.countplot(x='Churn', data=df)

plt.title("Churn Distribution")
plt.show()

In [ ]:
df.sample()

## Churn vs Contract Type

In [ ]:
sns.countplot(x='Contract_One year', hue='Churn', data=df)
plt.title("Churn vs Contract")
plt.show()

In [ ]:
sns.countplot(x='Contract_Two year', hue='Churn', data=df)

plt.title("Churn vs Contract")
plt.show()

## Churn vs Tenure

In [ ]:
sns.histplot(data=df, x='tenure', hue='Churn', bins=30)

plt.title("Churn vs Tenure")
plt.show()

## Churn vs Monthly Charges


In [ ]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)

plt.title("Churn vs Monthly Charges")
plt.show()

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

## Train-Test Split

We split the dataset into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Handling Imbalanced Data using SMOTE

The dataset is imbalanced, meaning there are more non-churn customers than churn customers.

SMOTE is used to balance the dataset by creating synthetic examples
of the minority class.

IMPORTANT:
SMOTE is applied only on the training data to avoid data leakage.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train
)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_resampled.value_counts())

## Logistic Regression

Logistic Regression is a classification algorithm used to predict
binary outcomes.

In this project, it predicts whether a customer will churn (1) or not (0).

It estimates the probability of churn using a logistic (sigmoid) function.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=100)
lr.fit(X_train_resampled,y_train_resampled)
y_pred_lr = lr.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

### Accuracy can be misleading in imbalanced datasets

## K-Nearest Neighbors (KNN)

KNN is a distance-based algorithm that classifies a data point
based on its nearest neighbors.

It can capture non-linear relationships better than Logistic Regression.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train_resampled, y_train_resampled)

y_pred_knn = knn.predict(X_test_scaled)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

### High recall alone is NOT enough

## Decision Tree Classifier

Decision Trees split the data based on feature values
and can capture complex patterns.

They often perform well on imbalanced datasets.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train_resampled, y_train_resampled)

y_pred_dt = dt.predict(X_test_scaled)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

## Random Forest Classifier

Random Forest is an ensemble model that combines multiple
decision trees to improve performance and reduce overfitting.

It is one of the most powerful models for classification problems.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)

rf.fit(X_train_resampled, y_train_resampled)

y_pred_rf = rf.predict(X_test_scaled)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

## Support Vector Machine (SVM)

SVM finds the optimal boundary that separates classes.

It works well for classification problems with complex boundaries.

In [ ]:
from sklearn.svm import SVC

svm = SVC()

svm.fit(X_train_resampled, y_train_resampled)

y_pred_svm = svm.predict(X_test_scaled)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

## Saving the Best Model

Decision Tree performed best for churn prediction,
so we save it using pickle.

In [54]:
import pickle

with open("./Model/churn_model.pkl", "wb") as file:
    pickle.dump(dt, file)

print("Model saved successfully!")

Model saved successfully!
